# 04 -- LGD model (loss given default) -- the key feature

**What this notebook does (plain English):** When a mortgage defaults, the lender
doesn't lose everything -- it sells the house and recovers most of the money.
**LGD** is the slice that is actually lost. Unlike a typical consumer-credit
project (where LGD is an assumption), here we model LGD from Freddie Mac's
**real, settled loss figures**. We use a simple **two-stage** model: the chance
of *any* loss, times the *size* of the loss when it happens.

**Headline result:** modelled LGD is roughly **double in the downturn** (~55%)
versus the calm year (~25%) -- a real, data-driven downturn LGD, which is the
single thing this project exists to demonstrate.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table; LGD is modelled ONLY on defaulted, disposed loans.
import pandas as pd
import numpy as np
from src.models import TwoStageLGD
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
disposed = base[base['disposed'] & base['lgd'].notna()].copy()
print('disposed defaults used for LGD:', len(disposed))

disposed defaults used for LGD: 6749


In [3]:
# Fit the two-stage LGD model (P(loss) x severity) and predict back on them.
lgd_model = TwoStageLGD().fit(disposed)
disposed['lgd_hat'] = lgd_model.predict(disposed)

In [4]:
# Compare observed vs modelled LGD, downturn (2007/2008) vs calm (2015), and
# show the three LGD lenses side by side: nominal IFRS 9, economic IFRS 9, APRA.
disposed['regime'] = np.where(disposed['vintage_year'].isin([2007, 2008]), 'downturn (2007-08)', 'calm (2015)')
tbl = disposed.groupby('regime').agg(
    disposed_defaults=('lgd', 'size'),
    observed_lgd=('lgd', 'mean'),
    modelled_lgd=('lgd_hat', 'mean'),
    observed_lgd_econ=('lgd_econ', 'mean'),
    lgd_apra=('lgd_apra', 'mean'),
).reset_index().round(4)

In [5]:
# Add an "all vintages" row and save as this notebook's result table.
overall = pd.DataFrame([{
    'regime': 'all', 'disposed_defaults': len(disposed),
    'observed_lgd': round(disposed['lgd'].mean(), 4),
    'modelled_lgd': round(disposed['lgd_hat'].mean(), 4),
    'observed_lgd_econ': round(disposed['lgd_econ'].mean(), 4),
    'lgd_apra': round(disposed['lgd_apra'].mean(), 4),
}])
lgd_summary = pd.concat([tbl, overall], ignore_index=True)
save_csv(lgd_summary, 'output/04_lgd_model.csv')
lgd_summary

,regime,disposed_defaults,observed_lgd,modelled_lgd,observed_lgd_econ,lgd_apra
0,calm (2015),136,0.2464,0.2325,0.3033,0.3635
1,downturn (2007-08),6613,0.5673,0.5664,0.6214,0.6340
2,all,6749,0.5608,0.5596,0.6150,0.6285


**Reading the table:** `observed_lgd` is what actually happened (nominal
IFRS 9); `modelled_lgd` is the two-stage model's fit. The downturn row sits roughly
twice as high as the calm row -- the **downturn LGD** a stress test needs.

The last two columns are the framework views built in notebook 01, carried through
here so a reviewer sees them next to the model:
- **`observed_lgd_econ`** -- the *economic* (discounted) IFRS 9 loss; >= nominal
  because the recovery is discounted over the workout (APS 113 Att D LGD para 1).
- **`lgd_apra`** -- the **APRA regulatory-capital view**: mortgage-insurance
  recoveries excluded (APS 113 Att B para 23), the 20% high-LVR+LMI reduction
  applied, then floored at 20% (APS 113 Att B paras 19-24). It is deliberately the
  most conservative column and is **never** mixed into the IFRS 9 figures.

The model is built only on loans that truly disposed, so every number is grounded
in a real settled loss.

In [6]:
# Cyclicality test (APS 113 Att D LGD paras 4-5): is loss severity materially
# higher in bad years than good? If so, a DOWNTURN LGD is required, not optional.
cyc = disposed.groupby('regime').agg(
    n=('lgd', 'size'), realised_lgd=('lgd', 'mean')).reset_index()
calm_lgd = float(cyc.loc[cyc['regime'].str.startswith('calm'), 'realised_lgd'].iloc[0])
down_lgd = float(cyc.loc[cyc['regime'].str.startswith('downturn'), 'realised_lgd'].iloc[0])
print('calm LGD     : {:.4f}'.format(calm_lgd))
print('downturn LGD : {:.4f}'.format(down_lgd))
print('downturn / calm ratio: {:.2f}x'.format(down_lgd / calm_lgd))
print('=> severity is strongly cyclical, so the LGD ESTIMATE must reflect downturn '
      'conditions (APS 113 Att D LGD para 4-5), not the through-the-cycle average.')

calm LGD     : 0.2464
downturn LGD : 0.5673
downturn / calm ratio: 2.30x
=> severity is strongly cyclical, so the LGD ESTIMATE must reflect downturn conditions (APS 113 Att D LGD para 4-5), not the through-the-cycle average.


**Cyclicality (P2-3).** Realised severity is far higher in the crisis books
than the calm one (roughly a 2x ratio), which is the textbook signature of a
**cyclical** LGD. Under APS 113 Att D LGD paras 4-5, where loss severity is cyclical
the LGD *estimate* used for capital/EL must reflect **downturn** conditions rather
than the long-run average. Notebook 06 therefore carries an explicit downturn-LGD
variant of Expected Loss alongside the through-the-cycle one.